# Ingest AG News into a LlamaStack Vector Store with Metadata

This notebook loads a small sample from the Hugging Face `fancyzhx/ag_news` dataset, creates a LlamaStack / OGX vector store, uploads each article as a separate file, and attaches metadata attributes that can later be used for filtering during retrieval.

The vector store is also created with vector-store-level metadata: `version_no` and `tenant_id`. The same values are added to each uploaded file's attributes so they can be used consistently for governance and filtering in multi-tenant or versioned RAG demos.

The ingestion pattern is useful for RAG demos where you want to test metadata filtering by fields such as `category`, ` or `document_type`.


## 1. Install dependencies

Run this cell once in a fresh environment. If your notebook environment already has these packages installed, you can skip it.

In [ ]:
# Uncomment and run if needed
%pip install datasets pandas llama-stack-client==0.6.1

## 2. Import libraries and configure connection

The notebook uses environment variables when available, with default values for local demo convenience.

- `LLAMASTACK_URL`: Base URL of your LlamaStack / OGX endpoint.
- `EMBED_MODEL`: Registered embedding model name in LlamaStack.
- `VECTOR_STORE_NAME`: Name for the new vector store.
- `DATASET_NAME`: Hugging Face dataset used for the demo.
- `TENANT_ID`: Tenant identifier stored as vector-store metadata and file attributes.
- `VERSION_NO`: Version identifier stored as vector-store metadata and file attributes.


In [ ]:
import os
import tempfile
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from llama_stack_client import LlamaStackClient

#optional define hugging face token
HF_TOKEN = os.getenv("HF_TOKEN","<<Hugging Face Token>>")
LLAMASTACK_URL = os.getenv(
    "LLAMASTACK_URL",
    "http://lsd-service.agnews-rag-demo.svc.cluster.local:8321",
)

VECTOR_STORE_NAME = os.getenv(
    "VECTOR_STORE_NAME",
    "ag_news",
)

EMBED_MODEL = os.getenv(
    "EMBED_MODEL",
    "sentence-transformers/ibm-granite/granite-embedding-125m-english",
)

# Vector-store-level metadata values.
# Override these through environment variables when running the notebook in OpenShift / Workbench.
TENANT_ID = os.getenv("TENANT_ID", "DEV")
VERSION_NO = os.getenv("VERSION_NO", "v1")

VECTOR_STORE_METADATA = {
    "tenant_id": TENANT_ID,
    "version_no": VERSION_NO,
}

DATASET_NAME = "fancyzhx/ag_news"
DATASET_REVISION = "eb185aade064a813bc0b7f42de02595523103ca4"
NU_ROWS_DATASET_LOAD = int(os.getenv("NU_ROWS_DATASET_LOAD", "10"))

client = LlamaStackClient(base_url=LLAMASTACK_URL)

print("LlamaStack URL:", LLAMASTACK_URL)
print("Vector store name:", VECTOR_STORE_NAME)
print("Embedding model:", EMBED_MODEL)
print("Vector store metadata:", VECTOR_STORE_METADATA)


## 3. Define label mapping and helper functions

The AG News dataset has integer labels. We map those labels to readable category names so they can be used as metadata filters later.

In [ ]:
LABEL_MAP = {
    0: "world",
    1: "sports",
    2: "business",
    3: "sci_tech",
}


def clean(value):
    """Return a stripped string value, or None for missing values."""
    if value is None or pd.isna(value):
        return None
    return str(value).strip()


def safe_int(value):
    """Convert a value to int safely, returning None if conversion fails."""
    if value is None or pd.isna(value):
        return None
    try:
        return int(float(value))
    except Exception:
        return None


def build_article_record(row, row_id):
    """Build the text payload and metadata attributes for one AG News row."""
    article = clean(row.get("text"))
    label_id = safe_int(row.get("label"))
    category = LABEL_MAP.get(label_id, "unknown")

    text = f"""
{article}
""".strip()

    attributes = {
        "doc_id": f"ag-news-{row_id}",
        "dataset": DATASET_NAME,
        "source": "huggingface",
        "category": category,
        "document_type": "news_article",
    }

    return {
        "text": text,
        "attributes": attributes,
    }


## 4. Load a small AG News sample

This demo ingests only the first `NU_ROWS_DATASET_LOAD` training records. Increase the environment variable, for example `NU_ROWS_DATASET_LOAD=100`, when you want a larger test set.


In [ ]:
SPLIT_NAME = os.getenv("SPLIT_NAME", "train")
dataset = load_dataset(DATASET_NAME, split=f"{SPLIT_NAME}[:{NU_ROWS_DATASET_LOAD}]",token=HF_TOKEN,revision=DATASET_REVISION)
df = pd.DataFrame(dataset).reset_index(drop=True)

print("Available columns:", list(df.columns))
print("Rows selected for ingestion:", len(df))

df.head()


## 5. Create the vector store with metadata

The vector store is created with the configured embedding model, the `milvus-remote` provider, and vector-store-level metadata containing `tenant_id` and `version_no`.

Adjust `provider_id` if your LlamaStack distribution uses a different vector I/O provider.


In [ ]:
vector_store = client.vector_stores.create(
    name=VECTOR_STORE_NAME,
    metadata=VECTOR_STORE_METADATA,
    extra_body={
        "embedding_model": EMBED_MODEL,
        "provider_id": "milvus-remote",
    },
)

VECTOR_STORE_ID = vector_store.id
print("Vector store created:", VECTOR_STORE_ID)
print("Vector store metadata:", VECTOR_STORE_METADATA)


## 6. Configure chunking strategy

Static chunking splits text into fixed-size token windows. The values below are practical defaults for small news articles:

- `max_chunk_size_tokens = 512`
- `chunk_overlap_tokens = 60`

For AG News, most articles are short, so each article will usually become one chunk.

In [ ]:
chunking_strategy = {
    "type": "static",
    "static": {
        "max_chunk_size_tokens": 512,
        "chunk_overlap_tokens": 60,
    },
}

chunking_strategy

## 7. Upload and attach each article

Each article is written to a temporary `.txt` file, uploaded through the Files API, then attached to the vector store with metadata attributes. These attributes become filterable fields during vector store search.

The file attributes include `tenant_id` and `version_no` as well. This keeps the data filterable at retrieval time while the vector store also carries the same high-level metadata.


In [ ]:
uploaded_files = []
attached_files = []
ingestion_results = []

for i, row in df.iterrows():
    record = build_article_record(row, i)
    attrs = record["attributes"]

    txt_file = tempfile.NamedTemporaryFile(
        mode="w",
        suffix=f"_ag_news_{i}.txt",
        delete=False,
        encoding="utf-8",
    )
    txt_path = txt_file.name

    with txt_file as f:
        f.write(record["text"])

    print(f"Uploading article {i}: {txt_path}")

    with open(txt_path, "rb") as file_handle:
        uploaded_file = client.files.create(
            file=file_handle,
            purpose="assistants",
        )

    uploaded_files.append(uploaded_file.id)

    print("File uploaded:", uploaded_file.id)
    print("Attaching with attributes:", attrs)

    vs_file = client.vector_stores.files.create(
        vector_store_id=VECTOR_STORE_ID,
        file_id=uploaded_file.id,
        attributes=attrs,
        chunking_strategy=chunking_strategy,
    )

    attached_files.append(vs_file.id)

    ingestion_results.append({
        "row_id": i,
        "file_id": uploaded_file.id,
        "vector_store_file_id": vs_file.id,
        **attrs,
    })

    print("Attached:", vs_file.id)

print("Done.")

## 8. Review ingestion summary

The table below gives you a quick audit trail for the uploaded files and their metadata.

In [ ]:
summary_df = pd.DataFrame(ingestion_results)

print("VECTOR_STORE_ID =", VECTOR_STORE_ID)
print("Vector store name =", VECTOR_STORE_NAME)
print("Rows ingested =", len(df))
print("Files uploaded =", len(uploaded_files))
print("Files attached =", len(attached_files))

summary_df

## 9. Optional: example metadata filters for later search

After ingestion, you can search this vector store and filter by metadata such as `category = business`, `tenant_id`, or `version_no`. The exact search method can vary by client version, but the filter shape usually follows the LlamaStack vector store search API.


In [ ]:
example_filter = {
    "type": "and",
    "filters": [
        {
            "type": "eq",
            "key": "category",
            "value": "business",
        },
        {
            "type": "eq",
            "key": "tenant_id",
            "value": TENANT_ID,
        },
        {
            "type": "eq",
            "key": "version_no",
            "value": VERSION_NO,
        },
    ],
}

example_search_body = {
    "query": "Find business news about oil prices",
    "filters": example_filter,
    "max_num_results": 5,
    "search_mode": "vector",
}

example_search_body
